# TSSTG 장소 1·장소 2·전체 학습 (seed 42)
Colab 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택한 뒤 위에서부터 순서대로 실행합니다. 장소별 모델 2개와 전체 모델 1개를 학습하고, 동일한 공통 Test 20개로 Raw TSSTG까지 비교합니다.

In [1]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요.'
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
Tesla T4, 15360 MiB


## 1. 학습 묶음 업로드
로컬의 `exports/tsstg_full200_colab_seed42_ready.zip`을 선택합니다.

In [4]:
from google.colab import files
# uploaded = files.upload()
# archive = next(iter(uploaded))
# print('uploaded:', archive)
# !rm -rf /content/tsstg_full200_seed42
!mkdir -p /content/tsstg_full200_seed42
!unzip -q -o "/content/tsstg_full200_colab_seed42_ready.zip" -d /content/tsstg_full200_seed42
%cd /content/tsstg_full200_seed42
!find . -maxdepth 2 -type f | sort | head -30

/content/tsstg_full200_seed42
./docs/TSSTG_200_EXPERIMENT_KO.md
./tsstg_pipeline/compare_models.py
./tsstg_pipeline/__init__.py
./tsstg_pipeline/train_pilot.py


In [5]:
import json
from pathlib import Path
for name in ['S001_location1_compare', 'S001_location2_compare', 'S001_full200_compare']:
    summary = json.loads((Path('tsstg_binary_dataset') / name / 'dataset_summary.json').read_text())
    print(name, summary['split_counts'], 'excluded=', summary['excluded_train_event_ids'])

S001_location1_compare {'train': 80, 'validation': 10, 'test': 10} excluded= []
S001_location2_compare {'train': 79, 'validation': 10, 'test': 10} excluded= ['S001_E101']
S001_full200_compare {'train': 159, 'validation': 20, 'test': 20} excluded= ['S001_E101']


## 2. 세 모델 학습
모두 원본 pretrained TSSTG에서 독립적으로 시작하며 seed는 42입니다.

In [6]:
!python -m tsstg_pipeline.train_pilot \
  --dataset tsstg_binary_dataset/S001_location1_compare \
  --output run_location1_seed42 --head-epochs 15 --finetune-epochs 30 \
  --temporal-dropout-probability 0.35 --temporal-dropout-max-frames 4 --seed 42

device=cuda
Loaded 340/342 compatible tensors
/content/tsstg_full200_seed42/tsstg_pipeline/train_pilot.py:122: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  total_loss += float(loss) * len(labels)
epoch=001 train_loss=0.6954 train_acc=0.500 val_loss=0.6523 val_acc=0.500
epoch=002 train_loss=0.6469 train_acc=0.775 val_loss=0.6158 val_acc=1.000
epoch=003 train_loss=0.6270 train_acc=0.762 val_loss=0.5889 val_acc=0.900
epoch=004 train_loss=0.5957 train_acc=0.825 val_loss=0.5621 val_acc=1.000
epoch=005 train_loss=0.5454 train_acc=0.912 val_loss=0.5382 val_acc=1.000
epoch=006 train_loss=0.5371 train_acc=0.838 val_loss=0.5140 val_acc=1.000
epoch=007 train_loss=0.5232 train_acc=0.863 val_loss=0.4935 val_acc=1.000
epoch=008 train_loss=0.5552 train_acc=0.787 val_loss=0.4705 val_acc=1.000
epoch=009 train_l

In [7]:
!python -m tsstg_pipeline.train_pilot \
  --dataset tsstg_binary_dataset/S001_location2_compare \
  --output run_location2_seed42 --head-epochs 15 --finetune-epochs 30 \
  --temporal-dropout-probability 0.35 --temporal-dropout-max-frames 4 --seed 42

device=cuda
Loaded 340/342 compatible tensors
/content/tsstg_full200_seed42/tsstg_pipeline/train_pilot.py:122: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  total_loss += float(loss) * len(labels)
epoch=001 train_loss=0.7277 train_acc=0.443 val_loss=0.7149 val_acc=0.500
epoch=002 train_loss=0.7048 train_acc=0.494 val_loss=0.7033 val_acc=0.500
epoch=003 train_loss=0.6570 train_acc=0.722 val_loss=0.7126 val_acc=0.400
epoch=004 train_loss=0.6426 train_acc=0.759 val_loss=0.7246 val_acc=0.500
epoch=005 train_loss=0.6094 train_acc=0.785 val_loss=0.7385 val_acc=0.500
epoch=006 train_loss=0.5868 train_acc=0.785 val_loss=0.7531 val_acc=0.500
epoch=007 train_loss=0.5635 train_acc=0.797 val_loss=0.7639 val_acc=0.500
epoch=008 train_loss=0.5445 train_acc=0.823 val_loss=0.7930 val_acc=0.600
epoch=009 train_l

In [8]:
!python -m tsstg_pipeline.train_pilot \
  --dataset tsstg_binary_dataset/S001_full200_compare \
  --output run_full200_seed42 --head-epochs 15 --finetune-epochs 30 \
  --temporal-dropout-probability 0.35 --temporal-dropout-max-frames 4 --seed 42

device=cuda
Loaded 340/342 compatible tensors
/content/tsstg_full200_seed42/tsstg_pipeline/train_pilot.py:122: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  total_loss += float(loss) * len(labels)
epoch=001 train_loss=0.7074 train_acc=0.472 val_loss=0.6830 val_acc=0.500
epoch=002 train_loss=0.6610 train_acc=0.610 val_loss=0.6539 val_acc=0.650
epoch=003 train_loss=0.5957 train_acc=0.786 val_loss=0.6543 val_acc=0.750
epoch=004 train_loss=0.5710 train_acc=0.780 val_loss=0.6689 val_acc=0.800
epoch=005 train_loss=0.5615 train_acc=0.774 val_loss=0.6545 val_acc=0.750
epoch=006 train_loss=0.5428 train_acc=0.767 val_loss=0.6441 val_acc=0.750
epoch=007 train_loss=0.5011 train_acc=0.792 val_loss=0.6351 val_acc=0.750
epoch=008 train_loss=0.5039 train_acc=0.799 val_loss=0.6300 val_acc=0.750
epoch=009 train_l

## 3. 동일한 공통 Test 20개로 평가

In [9]:
runs = {
    'location1': 'run_location1_seed42',
    'location2': 'run_location2_seed42',
    'full200': 'run_full200_seed42',
}
for name, run in runs.items():
    !python -m tsstg_pipeline.compare_models --dataset tsstg_binary_dataset/S001_full200_compare --binary-weights {run}/binary_tsstg_state_dict.pth --splits test --output {run}/common_test_comparison --device cuda

{
  "device": "cuda",
  "evaluated_splits": [
    "test"
  ],
  "data_files": [
    "/content/tsstg_full200_seed42/tsstg_binary_dataset/S001_full200_compare/test.npz"
  ],
  "samples": 20,
  "class_order": [
    "NON_FALL",
    "FALL"
  ],
  "confusion_matrix_convention": "rows=true, columns=predicted",
  "original_mapping": {
    "FALL": "7-class argmax == Fall Down",
    "NON_FALL": "7-class argmax is any other class"
  },
  "original_7class_tsstg": {
    "accuracy": 0.7,
    "balanced_accuracy": 0.7,
    "fall_precision": 1.0,
    "fall_recall": 0.4,
    "fall_f1": 0.5714285714285715,
    "non_fall_specificity": 1.0,
    "confusion_matrix": [
      [
        10,
        0
      ],
      [
        6,
        4
      ]
    ]
  },
  "fine_tuned_binary_tsstg": {
    "accuracy": 0.75,
    "balanced_accuracy": 0.75,
    "fall_precision": 0.7272727272727273,
    "fall_recall": 0.8,
    "fall_f1": 0.761904761904762,
    "non_fall_specificity": 0.7,
    "confusion_matrix": [
      [
        

In [10]:
import pandas as pd
rows = []
first_report = None
for name, run in runs.items():
    report = json.loads((Path(run) / 'common_test_comparison' / 'comparison_metrics.json').read_text())
    first_report = first_report or report
    metric = report['fine_tuned_binary_tsstg']
    rows.append({'model': name, 'scope': 'all_20', **{k: metric[k] for k in ['accuracy','balanced_accuracy','fall_precision','fall_recall','fall_f1','non_fall_specificity']}})
    for location, block in report['metrics_by_location'].items():
        metric = block['fine_tuned_binary_tsstg']
        rows.append({'model': name, 'scope': location, **{k: metric[k] for k in ['accuracy','balanced_accuracy','fall_precision','fall_recall','fall_f1','non_fall_specificity']}})
raw = first_report['original_7class_tsstg']
rows.insert(0, {'model': 'raw_tsstg', 'scope': 'all_20', **{k: raw[k] for k in ['accuracy','balanced_accuracy','fall_precision','fall_recall','fall_f1','non_fall_specificity']}})
for location, block in first_report['metrics_by_location'].items():
    metric = block['original_7class_tsstg']
    rows.append({'model': 'raw_tsstg', 'scope': location, **{k: metric[k] for k in ['accuracy','balanced_accuracy','fall_precision','fall_recall','fall_f1','non_fall_specificity']}})
results = pd.DataFrame(rows).sort_values(['scope','model']).reset_index(drop=True)
display(results.style.format(precision=3))
results.to_csv('seed42_model_comparison.csv', index=False, encoding='utf-8-sig')

,model,scope,accuracy,balanced_accuracy,fall_precision,fall_recall,fall_f1,non_fall_specificity
0,full200,all_20,0.800,0.800,0.800,0.800,0.800,0.800
1,location1,all_20,0.750,0.750,0.727,0.800,0.762,0.700
2,location2,all_20,0.600,0.600,0.583,0.700,0.636,0.500
3,raw_tsstg,all_20,0.700,0.700,1.000,0.400,0.571,1.000
4,full200,location_1,0.800,0.800,0.800,0.800,0.800,0.800
5,location1,location_1,0.800,0.800,0.800,0.800,0.800,0.800
6,location2,location_1,0.600,0.600,0.600,0.600,0.600,0.600
7,raw_tsstg,location_1,0.700,0.700,1.000,0.400,0.571,1.000
8,full200,location_2,0.800,0.800,0.800,0.800,0.800,0.800
9,location1,location_2,0.700,0.700,0.667,0.800,0.727,0.600


## 4. 결과 다운로드

In [11]:
!zip -qr tsstg_seed42_results.zip run_location1_seed42 run_location2_seed42 run_full200_seed42 seed42_model_comparison.csv
files.download('tsstg_seed42_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>